## Benchmarking with `sktime`

So far we have looked into

- baselines
- statistical models
- deep learning models
- foundation models and
- agentic workflows

On top of these there are new foundation model releases almost every few weeks claiming to be at the top of some irreproducible leaderboard.

As a practitioner, it gets difficult and overwhelming to consider the right model for your use case.

`sktime`'s benchmarking framework provides an easy interface

- to reliably compare different forecasting strategies on your own dataset
- get a leaderboard ranking out of benchmarking result
- analyse the result with post-hoc analyzers


## What goes in a benchmark

A benchmark experiment is defined across a cartesian product of tasks and estimators.

* A **task** consists of a dataset, a cross-validation splitting strategy, and one or more metrics.

* An **estimator** for this demonstration is a forecaster. However, `sktime` also supports time series classification and regression benchmarking.


## Setting up and running your first benchmark

Now that we know what goes in a benchmark, lets try benchmarking

- the forecasters we saw in the introduction
- on the airline series

Airline series is for demonstration, to add your own dataset, you can follow the instructions here [add a new dataset](https://www.sktime.net/docs/developer-guide/add-dataset/)


In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

from sktime.benchmarking.forecasting import ForecastingBenchmark
from sktime.datasets import Airline
from sktime.forecasting.compose import make_reduction
from sktime.forecasting.ets import AutoETS
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.theta import ThetaForecaster
from sktime.forecasting.chronos import ChronosForecaster
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError
from sktime.split import ExpandingWindowSplitter
from sktime.transformations.series.difference import Differencer

smape = MeanAbsolutePercentageError(symmetric=True)
cv = ExpandingWindowSplitter(initial_window=96, fh=12, step_length=12)

benchmark = ForecastingBenchmark()
benchmark.add(
    {
        "seasonal_naive": NaiveForecaster(strategy="last", sp=12),
        "autoets": AutoETS(auto=True, sp=12),
        "theta": ThetaForecaster(sp=12),
        "hgb": make_reduction(
            HistGradientBoostingRegressor(random_state=0),
            strategy="direct",
            window_length=12,
        ),
        "hgb_diff": Differencer(lags=1)
        * make_reduction(
            HistGradientBoostingRegressor(random_state=0),
            strategy="direct",
            window_length=12,
        ),
        "chronos_bolt_tiny": ChronosForecaster("amazon/chronos-bolt-tiny"),
    }
)
benchmark.add(Airline(), cv, smape)


To set up the benchmark,

* Instantiate `ForecastingBenchmark`

    * `ForecastingBenchmark` also accepts configurations for different parallelisation backends such as joblib, loky, dask, and ray.
    * Currently the computation for different folds can be parallelised but we plan to allow parallelisation across different estimator-task pairs in the future releases too.

* Add forecasters to the benchmark

    * Forecasters can be given an `id` to distinguish two methods with same name but different configurations

* Add tasks

    * For demonstration, we add one task consisting of airline series and we use `ExpandingWindowSplitter` as our CV strategy
    * `ExpandingWindowSplitter` creates multiple test-train windows instead of one while respecting the temporal order, for more reliable evaluation

In [9]:
results = benchmark.run()
results

W1121 03:08:42.866000 12637 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 101/101 [00:00<00:00, 4103.50it/s]
/opt/anaconda3/envs/work/lib/python3.12/site-packages/sktime/forecasting/chronos.py:656: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  self.model_pipeline, torch.Tensor(_y_i), prediction_length, self._config


,validation_id,model_id,MeanAbsolutePercentageError_fold_0_test,MeanAbsolutePercentageError_fold_1_test,MeanAbsolutePercentageError_fold_2_test,MeanAbsolutePercentageError_fold_3_test,MeanAbsolutePercentageError_mean,MeanAbsolutePercentageError_std,fit_time_fold_0_test,fit_time_fold_1_test,...,fit_time_fold_3_test,fit_time_mean,fit_time_std,pred_time_fold_0_test,pred_time_fold_1_test,pred_time_fold_2_test,pred_time_fold_3_test,pred_time_mean,pred_time_std,runtime_secs
0,[dataset=airline]_[cv_splitter=ExpandingWindow...,seasonal_naive,0.113814,0.032201,0.117614,0.105718,0.092337,0.040396,0.000814,0.000662,...,0.000481,0.000613,0.000158,0.014661,0.009296,0.008649,0.007201,0.009952,0.003259,0.010564
1,[dataset=airline]_[cv_splitter=ExpandingWindow...,autoets,0.048074,0.047954,0.085369,0.041247,0.055661,0.020061,0.381353,0.333975,...,0.386152,0.362238,0.025522,0.001001,0.000979,0.001045,0.000995,0.001005,0.000028,0.363242
2,[dataset=airline]_[cv_splitter=ExpandingWindow...,theta,0.061854,0.040061,0.107951,0.052319,0.065546,0.029644,0.002472,0.002155,...,0.002193,0.002242,0.000154,0.001294,0.001173,0.001107,0.001124,0.001175,0.000085,0.003417
3,[dataset=airline]_[cv_splitter=ExpandingWindow...,hgb,0.174358,0.114121,0.138196,0.164788,0.147866,0.027205,0.457242,0.533582,...,0.606283,0.555308,0.076208,0.023023,0.024173,0.022593,0.025260,0.023762,0.001201,0.579070
4,[dataset=airline]_[cv_splitter=ExpandingWindow...,hgb_diff,0.054205,0.063235,0.112757,0.042931,0.068282,0.030791,0.458306,0.487789,...,0.582705,0.510975,0.053147,0.025886,0.027722,0.024238,0.024864,0.025678,0.001523,0.536653
5,[dataset=airline]_[cv_splitter=ExpandingWindow...,chronos_bolt_tiny,0.058370,0.077780,0.065394,0.054996,0.064135,0.010075,2.292575,0.617972,...,0.618553,1.033122,0.839664,0.035341,0.015498,0.014822,0.016636,0.020574,0.009873,1.053696


* `run` fits and scores every model on every fold and return a statistical summary as a dataframe. 

    * Resulting dataframe contains scores, fit time, and predict time across every fold.
    * To write the results on hard-disk, pass a file path to run. The results can be written as CSV, JSON, or Parquet.

In [10]:
benchmark.leaderboard()

,model_id,MeanAbsolutePercentageError_rank,MeanAbsolutePercentageError,fit_time_s,predict_time_s
0,autoets,1.0,0.055661,0.362238,0.001005
1,chronos_bolt_tiny,2.0,0.064135,1.033122,0.020574
2,theta,3.0,0.065546,0.002242,0.001175
3,hgb_diff,4.0,0.068282,0.510975,0.025678
4,seasonal_naive,5.0,0.092337,0.000613,0.009952
5,hgb,6.0,0.147866,0.555308,0.023762


After every run, it can be useful to get a ranking of forecasters.

`benchmark.leaderboard` presents a leaderboard summary of the benchmark run

- it presents an average rank of forecasters across the datasets added to the benchmark
- accepts `datasets` and `scorers` to present rankings for individual or a list of datasets and scorers

NOTE: `benchmark.leaderboard` is unreleased and will be a part of sktime 1.4.0.

## Post-hoc analyzers

For this demonstration we benchmarked 5 methods on a single dataset.

As the experiments grow larger, you might want to perform post-hoc tests on their results to understand their significance.

`sktime` provides tools for

* Friedman test
* Nemenyi test
* Critical-difference diagram

These analyzers can directly consume the result artefacts produced by `benchmark.run`


## Can we run large scale benchmark experiments with sktime?

`sktime`'s benchmark runs are

* crash-safe

    * the results of each task-estimator is persisited as the experiment progresses, so in case of a crash, the result is not lost

* resumable

    * experiments can be run from where they were left off intentionally or in case of a crash

* in case of raises

    * the experiment does not stop if one of the methods fail
    * they get recorded in `benchmark.failed_experiments` for inspection later

Docs: [benchmarking](https://www.sktime.net/en/latest/api_reference/benchmarking.html)
